# Crafter Tabula Rasa JEPA World Model: Google Colab Training Pipeline

This notebook implements the complete **Joint Embedding Predictive Architecture (JEPA)** training pipeline for Crafter.
- **Zero Cheat Policy & Planning**: Operates solely on raw visual RGB pixel observations (64x64x3) without accessing internal game state (`inventory`, `player_pos`, `semantic`, etc.).
- **Anti-Collapse Self-Supervision**: Employs **VICReg** (Invariance, Variance Hinge, Covariance Decorrelation) ensuring non-collapse without pixel decoders.
- **Intrinsic Epistemic Curiosity**: 4-head predictor ensemble disagreement drives exploration and discovery of causal affordances.
- **Two-Tier Persistent Memory**: 70% rolling FIFO + 30% Curated Milestone Archive.
- **Preemption-Resilient Checkpointing**: Saves checkpoints to Google Drive (`/content/drive/MyDrive/crafter_jepa/checkpoints`).

### Cell 1: Environment Verification & Hardware Diagnostics

In [ ]:
# Cell 1: Environment Verification & Hardware Diagnostics
import os
import sys
import platform
import torch

print("=" * 70)
print("  SYSTEM & HARDWARE DIAGNOSTICS")
print("=" * 70)
print(f"Python Version:   {platform.python_version()} ({sys.executable})")
print(f"OS Platform:      {platform.system()} {platform.release()}")
print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version:     {torch.version.cuda}")
    print(f"Device Count:     {torch.cuda.device_count()}")
    print(f"Device Name:      {torch.cuda.get_device_name(0)}")
    vram_bytes = torch.cuda.get_device_properties(0).total_memory
    print(f"Total VRAM:       {vram_bytes / (1024**3):.2f} GB")
else:
    print("Running in CPU Mode (Hardware accelerator not detected).")
print("=" * 70)

### Cell 2: Google Drive Mounting & Persistent Checkpoint Storage

In [ ]:
# Cell 2: Google Drive Mounting & Namespace Isolation (Rule 1: Dynamic Auto-Creation)
import os
from pathlib import Path

try:
    from google.colab import drive
    print("[Colab Environment Detected] Mounting Google Drive...")
    drive.mount('/content/drive')
except ImportError:
    print("[Non-Colab / Local Environment] Running locally.")

# Target: /content/drive/MyDrive/learn-a-world/prometheus/checkpoints (or /content/drive/MyDrive/crafter_jepa/checkpoints)
if os.path.exists("/content/drive/MyDrive"):
    BASE_DRIVE = Path("/content/drive/MyDrive/learn-a-world")
    MODEL_SUBDIR = BASE_DRIVE / "prometheus"
    CHECKPOINT_DIR = MODEL_SUBDIR / "checkpoints_v2"
    LOGS_DIR = MODEL_SUBDIR / "logs_v2"
    SHARDS_DIR = MODEL_SUBDIR / "shards_v2"
else:
    CHECKPOINT_DIR = Path("./checkpoints_v2")
    LOGS_DIR = Path("./logs_v2")
    SHARDS_DIR = Path("./shards_v2")

# Rule 1: Dynamic Auto-Creation
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)
SHARDS_DIR.mkdir(parents=True, exist_ok=True)
print(f"[+] Model checkpoint directory ready: {CHECKPOINT_DIR.resolve()}")
print(f"[+] Logs directory ready: {LOGS_DIR.resolve()}")
print(f"[+] Shards directory ready: {SHARDS_DIR.resolve()}")

### Cell 3: Pip Install Dependencies


In [ ]:
# Cell 3: Pip Install Dependencies
import sys
import subprocess

dependencies = [
    "crafter",
    "gymnasium>=0.28.0",
    "matplotlib>=3.5.0",
]

print("Installing required dependencies...")
cmd = [sys.executable, "-m", "pip", "install", "-q"] + dependencies
result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode == 0:
    print("[+] All dependencies installed successfully.")
else:
    print("[!] Output during dependency installation:")
    print(result.stdout)
    print(result.stderr)

import numpy as np
if np.__version__.startswith("1."):
    print(f"[!] Detected legacy NumPy {np.__version__} in Python 3.13. Upgrading to NumPy 2.x...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "numpy>=2.0.0"], check=False)
    print("[IMPORTANT] NumPy was upgraded. Please click: Runtime -> Restart session, then re-run cells.")
else:
    print(f"[+] NumPy version {np.__version__} verified compatible with Python 3.13.")


### Cell 4: Repository Setup & Path Configuration

In [ ]:
# Cell 4: Repository Setup & Path Configuration
import os
import sys
import subprocess
from pathlib import Path

# In Google Colab, ensure repository is cloned and up to date
if os.path.exists("/content"):
    repo_dir = Path("/content/learn-a-world")
    if not (repo_dir / "prometheus" / "src").exists():
        print("[+] Cloning repository from GitHub...")
        subprocess.run(["git", "clone", "https://github.com/sahiltolani30/learn-a-world.git", "/content/learn-a-world"], check=False)
    else:
        print("[+] Repository present, pulling latest updates...")
        subprocess.run(["git", "-C", "/content/learn-a-world", "pull", "origin", "main"], check=False)

# Locate and configure project root path
cwd = Path(os.getcwd())
candidates = [
    Path("/content/learn-a-world/prometheus"),
    Path("/content/repo/prometheus"),
    Path("/content/prometheus"),
    cwd / "prometheus",
    cwd,
    cwd.parent,
]

REPO_ROOT = cwd
for candidate in candidates:
    if (candidate / "src" / "agent").exists():
        REPO_ROOT = candidate
        break

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

if 'src' in sys.modules:
    del sys.modules['src']
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('src.'):
        del sys.modules[mod_name]

print(f"[+] Repository root configured: {REPO_ROOT.resolve()}")

# Verify core module imports
try:
    from src.env.wrapper import make_crafter_env
    from src.agent.agent import JEPAAgent
    from src.models.losses import VICRegLoss
    from src.memory.buffer import TwoTierReplayBuffer
    from src.memory.storage import CheckpointManager
    print("[+] Core JEPA modules imported successfully!")
except ImportError as err:
    print(f"[!] Import error: {err}")
    raise

### Cell 5: Fast CPU Smoke Test Verification (<60s)

In [ ]:
# Cell 5: Fast CPU Smoke Test Verification (<60s)
# Validates complete 13-stage JEPA loop (zero-cheat, VICReg, buffer, curiosity)
from src.test_smoke import run_smoke_test

print("Executing fast local CPU smoke test...")
smoke_results = run_smoke_test()
print(f"\n[+] CPU Smoke Test Passed! Elapsed: {smoke_results['total_elapsed_seconds']:.3f} s")

### Cell 6: Full GPU Training Run with Drive Checkpointing

In [ ]:
# Cell 6: Full GPU Training Run with Drive Checkpointing
import torch
from src.train import TrainingPipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Starting Tabula Rasa JEPA Training on device: {device}")

# Root Cause Fix (C1-C7) Configuration for Fresh Tabula Rasa Run
training_config = {
    "device": device,
    "total_steps": 25_000,
    "batch_size": 128,
    "checkpoint_dir": CHECKPOINT_DIR,
    "eval_interval": 500,
    "eval_episodes": 3,
    "seed": 42,
    "lr": 1e-4,
    "critic_lr": 3e-4,
    "warmup_steps": 100,
    "save_interval": 1_000,
    "no_resume": True,  # Fresh run to prevent fossilized Q ~ 22 interference
    "tier2_ratio": 0.30,
    # C1: Critic learns pure external reward (eliminates fossilized curiosity inflation)
    "critic_intrinsic_weight": 0.0,
    # C3: 5-step TD target for credit assignment across chop-chop-place chains
    "nstep": 5,
    # C4.1: Per-action normalized curiosity in actor policy prior
    "actor_intrinsic_weight": 0.1,
    # C4.2: Episodic k-NN novelty modulation for adaptive exploration
    "eps_base": 0.05,
    "eps_novelty_coef": 0.20,
    "eps_max": 0.50,
    # C2.3: Milestone-prioritized sampling (25% T1, 80% T2)
    "milestone_t1_priority": 0.25,
    "milestone_t2_priority": 0.80,
    # C5: Optimized update interval and action repeat
    "update_every": 2,
    "action_repeat": 1,
    "gamma": 0.99,
    "tau": 0.99,
    "log_interval": 100,
}

pipeline = TrainingPipeline(training_config)
train_results = pipeline.train()

print("")
print("[+] Training session completed successfully.")
print(f"Total Steps:        {train_results['total_steps']}")
print(f"Total Episodes:     {train_results['total_episodes']}")
print(f"Achievements Count: {len(train_results['achievements_unlocked'])}")
print(f"Achievements List:  {train_results['achievements_unlocked']}")
print(f"Success Counts:     {train_results['success_counts']}")
print(f"Final Checkpoint:   {train_results['final_checkpoint']}")


### Cell 7: Visualization and Evaluation
Plots Loss Curves (VICReg Invariance, Variance, Covariance), Curiosity Progression, and Crafter Achievement Unlock Curves.

In [ ]:
# Cell 7: Visualization and Evaluation
import matplotlib.pyplot as plt
import numpy as np

loss_history = pipeline.loss_history
achievement_history = pipeline.achievement_history

fig, axes = plt.subplots(3, 2, figsize=(16, 14))
fig.suptitle("Crafter Tabula Rasa JEPA Consolidation & Training Telemetry", fontsize=16, fontweight="bold")

steps_arr = np.arange(len(loss_history)) if loss_history else []

# 1. World Model Loss & VICReg Components
ax1 = axes[0, 0]
if loss_history:
    wm_losses = [m.get("wm_loss", 0.0) for m in loss_history]
    inv_losses = [m.get("invariance", 0.0) for m in loss_history]
    var_losses = [m.get("variance", 0.0) for m in loss_history]
    cov_losses = [m.get("covariance", 0.0) for m in loss_history]
    ax1.plot(steps_arr, wm_losses, label="Total WM Loss", color="#1f77b4", linewidth=1.5)
    ax1.plot(steps_arr, inv_losses, label="Invariance", color="#2ca02c", alpha=0.7)
    ax1.plot(steps_arr, var_losses, label="Variance Hinge", color="#d62728", alpha=0.7)
    ax1.plot(steps_arr, cov_losses, label="Covariance", color="#9467bd", alpha=0.7)
    ax1.set_title("World Model Loss & VICReg Decomposition")
    ax1.set_xlabel("Update Step")
    ax1.set_ylabel("Loss")
    ax1.grid(True, linestyle="--", alpha=0.5)
    ax1.legend()
else:
    ax1.text(0.5, 0.5, "No loss history recorded", ha="center", va="center")

# 2. Critic MSE Loss & Pure External Value Grounding (C1)
ax2 = axes[0, 1]
if loss_history:
    critic_losses = [m.get("critic_loss", 0.0) for m in loss_history]
    q_ext = [m.get("q_mean_ext", m.get("q_mean", 0.0)) for m in loss_history]
    ax2.plot(steps_arr, critic_losses, label="Critic MSE Loss", color="#ff7f0e", alpha=0.8)
    ax2.plot(steps_arr, q_ext, label="Expected External Q-Value", color="#17becf", alpha=0.8)
    ax2.set_title("Critic Optimization (Pure External Target)")
    ax2.set_xlabel("Update Step")
    ax2.set_ylabel("Value")
    ax2.grid(True, linestyle="--", alpha=0.5)
    ax2.legend()
else:
    ax2.text(0.5, 0.5, "No critic history recorded", ha="center", va="center")

# 3. Q-Spread Telemetry (C7: Flat Landscape Healing Invariant)
ax3 = axes[1, 0]
if loss_history:
    q_spreads = [m.get("q_spread", 0.0) for m in loss_history]
    ax3.plot(steps_arr, q_spreads, label="Q-Spread (Max Q - Min Q)", color="#8c564b", linewidth=1.5)
    ax3.axhline(y=0.5, color="red", linestyle="--", alpha=0.6, label="Healing Gate (0.5)")
    ax3.set_title("Action Q-Spread Telemetry (Healing Gate > 0.5)")
    ax3.set_xlabel("Update Step")
    ax3.set_ylabel("Q-Spread")
    ax3.grid(True, linestyle="--", alpha=0.5)
    ax3.legend()
else:
    ax3.text(0.5, 0.5, "No Q-spread recorded", ha="center", va="center")

# 4. Adaptive Exploration Signals (C4: Novelty Modulation)
ax4 = axes[1, 1]
if loss_history:
    eps_vals = [m.get("eps_mean", 0.0) for m in loss_history]
    exp_ints = [m.get("expected_int_mean", 0.0) for m in loss_history]
    ax4.plot(steps_arr, eps_vals, label="Episodic Novelty Epsilon", color="#e377c2", linewidth=1.5)
    ax4.plot(steps_arr, exp_ints, label="Expected Normalized Curiosity", color="#7f7f7f", alpha=0.7)
    ax4.set_title("Exploration Signals (Episodic Novelty & Normalized Curiosity)")
    ax4.set_xlabel("Update Step")
    ax4.set_ylabel("Metric Value")
    ax4.grid(True, linestyle="--", alpha=0.5)
    ax4.legend()
else:
    ax4.text(0.5, 0.5, "No exploration history recorded", ha="center", va="center")

# 5. Dual Evaluation Return Progression (C6: Greedy vs Stochastic)
ax5 = axes[2, 0]
if achievement_history:
    eval_steps = sorted(achievement_history.keys())
    greedy_rets = [achievement_history[s].get("greedy_mean_return", achievement_history[s].get("mean_return", 0.0)) for s in eval_steps]
    stoch_rets = [achievement_history[s].get("stochastic_mean_return", 0.0) for s in eval_steps]
    ax5.plot(eval_steps, greedy_rets, label="Greedy Mean Return", color="#2ca02c", marker="o", linewidth=2)
    ax5.plot(eval_steps, stoch_rets, label="Stochastic Mean Return", color="#1f77b4", marker="s", linestyle="--", linewidth=2)
    ax5.set_title("Dual Evaluation Episodic Return (Greedy vs Stochastic)")
    ax5.set_xlabel("Environment Steps")
    ax5.set_ylabel("Mean Return")
    ax5.grid(True, linestyle="--", alpha=0.5)
    ax5.legend()
else:
    ax5.text(0.5, 0.5, "No evaluation history recorded", ha="center", va="center")

# 6. Dual Evaluation Achievement Count Progression (C6)
ax6 = axes[2, 1]
if achievement_history:
    eval_steps = sorted(achievement_history.keys())
    greedy_achs = [achievement_history[s].get("greedy_achievement_count", achievement_history[s].get("achievement_count", 0)) for s in eval_steps]
    stoch_achs = [achievement_history[s].get("stochastic_achievement_count", 0) for s in eval_steps]
    ax6.plot(eval_steps, greedy_achs, label="Greedy Achievements Unlocked", color="#d62728", marker="o", linewidth=2)
    ax6.plot(eval_steps, stoch_achs, label="Stochastic Achievements Unlocked", color="#9467bd", marker="^", linestyle="--", linewidth=2)
    ax6.set_title("Dual Evaluation Achievements Unlocked (Greedy vs Stochastic)")
    ax6.set_xlabel("Environment Steps")
    ax6.set_ylabel("Unique Achievements")
    ax6.grid(True, linestyle="--", alpha=0.5)
    ax6.legend()
else:
    ax6.text(0.5, 0.5, "No evaluation history recorded", ha="center", va="center")

plt.tight_layout()
plt.savefig("crafter_jepa_consolidation_curves.png", dpi=300)
plt.show()
print("[+] Telemetry visualization generated and saved to crafter_jepa_consolidation_curves.png")



### Cell 8: Automated Stage-A Gate Checklist & Forensic Verification Report (§9 of ROOT_CAUSE_FIX_PLAN.md)


In [ ]:
# Cell 8: Automated Stage-A Gate Checklist & Forensic Verification Report (§9 of ROOT_CAUSE_FIX_PLAN.md)
import numpy as np

print("=" * 80)
print("  * STAGE-A GATE CHECKLIST & FORENSIC VERIFICATION REPORT")
print("=" * 80)

loss_history = pipeline.loss_history
ach_history = pipeline.achievement_history

# 1. Gate 1: NaN check & Mean Critic Loss over last 1,000 updates (< 0.3)
recent_losses = loss_history[-1000:] if len(loss_history) >= 1000 else loss_history
critic_losses = [m.get("critic_loss", 0.0) for m in recent_losses]
has_nans = any(np.isnan(c) or np.isinf(c) for c in critic_losses)
mean_critic = float(np.mean(critic_losses)) if critic_losses else 0.0
g1_pass = (not has_nans) and (mean_critic < 0.30)
print("[GATE 1] NaN Check & Final Critic Loss:")
print(f"         Has NaNs:     {has_nans}")
print(f"         Mean Critic:  {mean_critic:.4f} (Gate: < 0.30)")
status_g1 = "PASS" if g1_pass else "FAIL"
print(f"         STATUS:       {status_g1}")
print()

# 2. Gate 2: Q-Spread at end of run and trend (> 0.5 and growing)
q_spreads = [m.get("q_spread", 0.0) for m in loss_history]
first_5k = q_spreads[:min(len(q_spreads), 5000)]
last_5k = q_spreads[-min(len(q_spreads), 5000):]
m_first = float(np.mean(first_5k)) if first_5k else 0.0
m_last = float(np.mean(last_5k)) if last_5k else 0.0
final_q_spread = q_spreads[-1] if q_spreads else 0.0
g2_pass = (final_q_spread > 0.5) and (m_last > m_first)
print("[GATE 2] Q-Spread & Flat Landscape Healing:")
print(f"         Final Q-Spread: {final_q_spread:.4f} (Gate: > 0.50)")
print(f"         Trend:          First 5k: {m_first:.4f} -> Last 5k: {m_last:.4f}")
status_g2 = "PASS" if g2_pass else "FAIL"
print(f"         STATUS:         {status_g2}")
print()

# 3. Gate 3: Exploration Health & Funnel
total_eps = max(1, pipeline.current_episode)
sc = pipeline.success_counts
do_count = sc.get("do", 0)
table_count = sc.get("place_table", 0)
print("[GATE 3] Exploration Health & Success Counters:")
print(f"         Total Episodes Completed: {total_eps}")
print(f"         Success Counts Dict:      {sc}")
print(f"         do Successes:             {do_count}")
print(f"         place_table Successes:    {table_count}")
status_g3 = "PASS" if table_count > 0 and do_count > 0 else "WARNING"
print(f"         STATUS:                   {status_g3}")
print()

# 4. Gate 4: Total Milestones & Milestone Sampling Fraction
total_ms = pipeline.total_milestones
ms_fractions = []
for _ in range(min(100, max(1, len(loss_history)))):
    b = pipeline.buffer.sample(batch_size=min(64, pipeline.buffer.size))
    if "is_milestone" in b:
        ms_fractions.append(float(b["is_milestone"].float().mean().item()))
avg_ms_fraction = float(np.mean(ms_fractions)) if ms_fractions else 0.0
g4_pass = avg_ms_fraction >= 0.08
print("[GATE 4] Milestone Sampling Fraction:")
print(f"         Total Milestones Flagged: {total_ms}")
print(f"         Milestone Draw Fraction:  {avg_ms_fraction * 100:.2f}% (Gate: >= 8.0%)")
status_g4 = "PASS" if g4_pass else "FAIL"
print(f"         STATUS:                   {status_g4}")
print()

# 5. Gate 5: Dual Eval Modes at Final Eval Cycle
if ach_history:
    final_step = sorted(ach_history.keys())[-1]
    final_eval = ach_history[final_step]
    g_ret = final_eval.get("greedy_mean_return", final_eval.get("mean_return", 0.0))
    s_ret = final_eval.get("stochastic_mean_return", 0.0)
    g_achs = final_eval.get("greedy_achievements", [])
    s_achs = final_eval.get("stochastic_achievements", [])
    total_unique = len(pipeline.all_unlocked_achievements)
    g5_pass = total_unique >= 5 and (len(s_achs) >= len(g_achs))
    print(f"[GATE 5] Dual Evaluation Metrics @ Step {final_step}:")
    print(f"         Greedy Return:     {g_ret:.2f} | Achievements ({len(g_achs)}): {g_achs}")
    print(f"         Stochastic Return: {s_ret:.2f} | Achievements ({len(s_achs)}): {s_achs}")
    print(f"         Cumulative Unique: {total_unique} / 22")
    status_g5 = "PASS" if g5_pass else "FAIL"
    print(f"         STATUS:            {status_g5}")
    print()

print("=" * 80)


### Cell 9: Tracking Metric Introspection — Scene (b) Wood=2, Facing Empty Grass (§6.3)


In [ ]:
# Cell 9: Tracking Metric Introspection - Scene (b) Wood=2, Facing Empty Grass (§6.3)
import crafter
from src.diagnostics import CRAFTER_ACTION_NAMES

print("=" * 80)
print("  * TRACKING METRIC INTROSPECTION (SCENE B: WOOD=2, FACING GRASS)")
print("=" * 80)

env = crafter.Env(seed=123)
pipeline.agent.eval()

# Construct Scene (b): Facing grass with Wood=2 in inventory
obs_b = env.reset()
p_x, p_y = env._player.pos
env._world[p_x + 1, p_y] = "grass"
env._player.facing = (1, 0)
env._player.inventory["wood"] = 2
obs_b = env.render()

# Prepare tensor and forward pass
x_b = torch.from_numpy(obs_b).permute(2, 0, 1).unsqueeze(0).float().to(pipeline.device) / 255.0
with torch.no_grad():
    z_b = pipeline.agent.encoder(x_b)
    q1_b, q2_b = pipeline.agent.critic(z_b)
    q_vals_b = torch.min(q1_b, q2_b)[0].cpu().numpy()
    probs_b = torch.softmax(pipeline.agent.actor(z_b), dim=-1)[0].cpu().numpy()

# Rank all 17 actions by Q-value
sorted_indices = np.argsort(-q_vals_b)
act_pt = CRAFTER_ACTION_NAMES.index("place_table")
rank_pt = int(np.where(sorted_indices == act_pt)[0][0]) + 1

print("All 17 Actions Ranked by Critic Q-Value in Scene (b):")
print("-" * 75)
print(f" {'Rank':<6} | {'Action Name':<20} | {'Q-Value':<10} | {'Policy Prob':<10}")
print("-" * 75)
for rank, idx in enumerate(sorted_indices, start=1):
    marker = " <=== [TARGET ACTION]" if idx == act_pt else ""
    print(f" #{rank:<5} | {CRAFTER_ACTION_NAMES[idx]:<20} | {q_vals_b[idx]:+8.4f}   | {probs_b[idx]:.4f}{marker}")
print("-" * 75)
print()
print("[KEY CONSOLIDATION TRACKING METRIC]:")
print(f"    Rank of place_table: #{rank_pt} / 17")
print(f"    Q(place_table):      {q_vals_b[act_pt]:+.4f}")
print(f"    Target Gate (Stage B): Rank <= 3")
print(f"    Pre-Fix Baseline:      Rank #5 (below sleep, place_stone)")
if rank_pt <= 3:
    print(f"    RESULT:                EXCELLENT! Consolidation fully established (Rank #{rank_pt} <= 3)")
elif rank_pt < 5:
    print(f"    RESULT:                PROGRESS CONFIRMED! Improved from Rank #5 to Rank #{rank_pt}")
else:
    print(f"    RESULT:                Rank #{rank_pt} (Track consolidation progression across training)")
print("=" * 80)


### Cell 10: Official 50-Game Crafter Benchmark Evaluation (Greedy & Stochastic)


In [ ]:
# Cell 10: Official 50-Game Crafter Benchmark Evaluation (Greedy & Stochastic)
from src.diagnostics import AgentForensics
import matplotlib.pyplot as plt

forensics = AgentForensics(pipeline.agent)

print("=" * 70)
print("  * RUNNING OFFICIAL CRAFTER BENCHMARK (50 GAMES - GREEDY)")
print("=" * 70)
greedy_results = forensics.run_benchmark(num_episodes=50, stochastic=False)
print(f"Official Crafter Score (Greedy): {greedy_results['crafter_score']:.2f}%")
print(f"Average Achievements/Game:       {greedy_results['mean_achievements_per_game']:.2f}")
print(f"Average Survival Steps:          {greedy_results['mean_length']:.1f} +- {greedy_results['std_length']:.1f}")
print(f"Average Return:                  {greedy_results['mean_return']:.2f} +- {greedy_results['std_return']:.2f}")
print(f"Total Unique Achievements:       {greedy_results['total_unique_achievements']} / 22")
print(f"Unlocked Achievements:           {greedy_results['unlocked_list']}")
print()
print("=" * 70)
print("  * RUNNING OFFICIAL CRAFTER BENCHMARK (50 GAMES - STOCHASTIC)")
print("=" * 70)
stoch_results = forensics.run_benchmark(num_episodes=50, stochastic=True)
print(f"Official Crafter Score (Stochastic): {stoch_results['crafter_score']:.2f}%")
print(f"Average Achievements/Game:           {stoch_results['mean_achievements_per_game']:.2f}")
print(f"Average Survival Steps:              {stoch_results['mean_length']:.1f} +- {stoch_results['std_length']:.1f}")
print(f"Average Return:                      {stoch_results['mean_return']:.2f} +- {stoch_results['std_return']:.2f}")
print(f"Total Unique Achievements:           {stoch_results['total_unique_achievements']} / 22")
print(f"Unlocked Achievements:               {stoch_results['unlocked_list']}")

fig = forensics.plot_benchmark_results(stoch_results, save_path="crafter_stochastic_benchmark.png")
plt.show()
